## Chroma

### Chroma is an AI-Native open-source vector db focused on developer productivity and happiness. Chroma is licensed under Apache2.0

In [1]:
## Building a sample vectordb

from langchain_chroma import Chroma
from langchain_community.document_loaders import TextLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

c:\Users\yaswa\Downloads\GEN_AI\langchain-demo\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
loader = TextLoader("speech.txt")
data = loader.load()
data

[Document(metadata={'source': 'speech.txt'}, page_content="\nFive score years ago, a great American, in whose symbolic shadow we stand today, signed the Emancipation Proclamation. This momentous decree came as a great beacon light of hope to millions of Negro slaves who had been seared in the flames of withering injustice. It came as a joyous daybreak to end the long night of their captivity.\n\nBut one hundred years later, the Negro still is not free. One hundred years later, the life of the Negro is still sadly crippled by the manacles of segregation and the chains of discrimination. One hundred years later, the Negro lives on a lonely island of poverty in the midst of a vast ocean of material prosperity. One hundred years later, the Negro is still languished in the corners of American society and finds himself an exile in his own land. And so we've come here today to dramatize a shameful condition.\n\nIn a sense we've come to our nation's capital to cash a check. When the architects

In [3]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=0)
splits = text_splitter.split_documents(data)
splits

[Document(metadata={'source': 'speech.txt'}, page_content='Five score years ago, a great American, in whose symbolic shadow we stand today, signed the Emancipation Proclamation. This momentous decree came as a great beacon light of hope to millions of Negro slaves who had been seared in the flames of withering injustice. It came as a joyous daybreak to end the long night of their captivity.'),
 Document(metadata={'source': 'speech.txt'}, page_content="But one hundred years later, the Negro still is not free. One hundred years later, the life of the Negro is still sadly crippled by the manacles of segregation and the chains of discrimination. One hundred years later, the Negro lives on a lonely island of poverty in the midst of a vast ocean of material prosperity. One hundred years later, the Negro is still languished in the corners of American society and finds himself an exile in his own land. And so we've come here today to dramatize a"),
 Document(metadata={'source': 'speech.txt'}, 

In [6]:
hf_embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_db=Chroma.from_documents(splits,hf_embeddings)
vector_db

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11807.84it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
### Query

query = "I have a dream that one day this"

docs = vector_db.similarity_search(query)
docs

[Document(id='2252218d-9d5b-46f3-ae6f-de078e8b1dfc', metadata={'source': 'speech.txt'}, page_content="I have a dream that one day, down in Alabama, with its vicious racists, with its governor having his lips dripping with the words of 'interposition' and 'nullification' -- one day right there in Alabama little black boys and black girls will be able to join hands with little white boys and white girls as sisters and brothers.\n\nI have a dream today!"),
 Document(id='59c5a102-af1c-4ca2-a540-4b36282ba12b', metadata={'source': 'speech.txt'}, page_content="I have a dream that one day, down in Alabama, with its vicious racists, with its governor having his lips dripping with the words of 'interposition' and 'nullification' -- one day right there in Alabama little black boys and black girls will be able to join hands with little white boys and white girls as sisters and brothers.\n\nI have a dream today!"),
 Document(id='8f4d24e3-142c-4949-bbf6-fbc3e152b1a1', metadata={'source': 'speech.txt

In [10]:
docs[0].page_content

"I have a dream that one day, down in Alabama, with its vicious racists, with its governor having his lips dripping with the words of 'interposition' and 'nullification' -- one day right there in Alabama little black boys and black girls will be able to join hands with little white boys and white girls as sisters and brothers.\n\nI have a dream today!"

In [ ]:
retriever = vector_db.as_retriever()
docs_new = retriever.invoke(query)  # docs_new = retriever.invoke(query)[0].page_content
docs_new[0]

[Document(id='2252218d-9d5b-46f3-ae6f-de078e8b1dfc', metadata={'source': 'speech.txt'}, page_content="I have a dream that one day, down in Alabama, with its vicious racists, with its governor having his lips dripping with the words of 'interposition' and 'nullification' -- one day right there in Alabama little black boys and black girls will be able to join hands with little white boys and white girls as sisters and brothers.\n\nI have a dream today!"),
 Document(id='59c5a102-af1c-4ca2-a540-4b36282ba12b', metadata={'source': 'speech.txt'}, page_content="I have a dream that one day, down in Alabama, with its vicious racists, with its governor having his lips dripping with the words of 'interposition' and 'nullification' -- one day right there in Alabama little black boys and black girls will be able to join hands with little white boys and white girls as sisters and brothers.\n\nI have a dream today!"),
 Document(id='8f4d24e3-142c-4949-bbf6-fbc3e152b1a1', metadata={'source': 'speech.txt

In [11]:
### saving to the disk

vector_db=Chroma.from_documents(splits,hf_embeddings,persist_directory='./chroma_db')
vector_db

In [12]:
### load from disk

vector_db_load = Chroma(persist_directory='./chroma_db',embedding_function=hf_embeddings)

In [15]:
docs = vector_db_load.similarity_search(query)
print(docs[0].page_content)

I have a dream that one day, down in Alabama, with its vicious racists, with its governor having his lips dripping with the words of 'interposition' and 'nullification' -- one day right there in Alabama little black boys and black girls will be able to join hands with little white boys and white girls as sisters and brothers.

I have a dream today!


In [17]:
docs = vector_db_load.similarity_search_with_score(query)
docs

[(Document(id='94261ac6-d774-4782-9154-da17891747f3', metadata={'source': 'speech.txt'}, page_content="I have a dream that one day, down in Alabama, with its vicious racists, with its governor having his lips dripping with the words of 'interposition' and 'nullification' -- one day right there in Alabama little black boys and black girls will be able to join hands with little white boys and white girls as sisters and brothers.\n\nI have a dream today!"),
  0.7747812271118164),
 (Document(id='3f15b798-bab0-423e-a4c8-4792941e8d10', metadata={'source': 'speech.txt'}, page_content='I have a dream that one day on the red hills of Georgia, the sons of former slaves and the sons of former slave owners will be able to sit down together at the table of brotherhood.\n\nI have a dream that one day even the state of Mississippi, a state sweltering with the heat of injustice, sweltering with the heat of oppression, will be transformed into an oasis of freedom and justice.'),
  0.7983462810516357),
